In [ ]:
import cv2
import mediapipe as mp
import csv
import os

# 1. ตั้งค่า MediaPipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(
    static_image_mode=False,        
    max_num_hands=1, # เก็บทีละมือเพื่อให้นิ่งที่สุด               
    min_detection_confidence=0.7,   
    min_tracking_confidence=0.7     
)

csv_filename = "asl_dvanced_dataset.csv"

# สร้างหัวตารางใหม่ (ตัด wrist ออก เพราะหักลบตัวเองได้ 0)
if not os.path.exists(csv_filename):
    with open(csv_filename, mode='w', newline='') as f:
        writer = csv.writer(f)
        header = ['label']
        for i in range(1, 21): # เก็บจุดที่ 1 ถึง 20
            header.extend([f'pt{i}_x', f'pt{i}_y', f'pt{i}_z'])
        writer.writerow(header)

cap = cv2.VideoCapture(0)
print("=== ระบบบันทึกพิกัดมือเชิงลึก (Relative 21 Points) ===")
print("วิธีใช้: ทำท่าค้างไว้แล้วกดปุ่มอักษร (A-Z) บนคีย์บอร์ดค้างไว้ | กด 'ESC' เพื่อปิด")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    h, w, c = frame.shape
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    key = cv2.waitKey(1) & 0xFF
    if key == 27: break
    pressed_char = chr(key).upper() if (97 <= key <= 122 or 65 <= key <= 90) else None

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # ดึงพิกัดข้อมือ (ID 0) เพื่อใช้เป็นจุดศูนย์กลาง (0, 0, 0)
            wrist = hand_landmarks.landmark[0]
            
            features_to_save = []
            # วนลูปเก็บจุดที่ 1 ถึง 20 โดยนำไปลบกับพิกัดข้อมือ
            for i in range(1, 21):
                lm = hand_landmarks.landmark[i]
                
                # ทำ Relative Coordinates (คำนวณระยะห่างสัมพัทธ์จากข้อมือ)
                rel_x = lm.x - wrist.x
                rel_y = lm.y - wrist.y
                rel_z = lm.z - wrist.z # แกน Z สัมพัทธ์ จะบอกว่านิ้วอยู่หน้าหรือหลังข้อมือ
                
                features_to_save.extend([rel_x, rel_y, rel_z])
            
            if pressed_char:
                with open(csv_filename, mode='a', newline='') as f:
                    writer = csv.writer(f)
                    writer.writerow([pressed_char] + features_to_save)
                
                cv2.putText(frame, f"SAVING: {pressed_char}", (20, h - 20), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow('Advanced Data Collector', frame)

cap.release()
cv2.destroyAllWindows()

=== ระบบบันทึกพิกัดมือเชิงลึก (Relative 21 Points) ===
วิธีใช้: ทำท่าค้างไว้แล้วกดปุ่มอักษร (A-Z) บนคีย์บอร์ดค้างไว้ | กด 'ESC' เพื่อปิด


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pickle

# 1. โหลดข้อมูลจากไฟล์ CSV ของคุณ
dataset_path = 'asl_dvanced_dataset.csv'
data = pd.read_csv(dataset_path)
data.dropna(inplace=True)  # ลบแถวที่มีค่า NaN ออก

print(f" Loaded data: {len(data)} rows")

# 2. แยกตัวแปร Features (พิกัดมือ) และ Label (ตัวอักษร)
X = data.drop('label', axis=1).values 
y = data['label'].values              

# 3. แบ่งข้อมูลเอาไว้สอน 80% และเอาไว้ทดสอบความแม่นยำ 20%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(" Training AI Model...")

# 4. ใช้สมองกลแบบ Random Forest ในการเรียนรู้
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# 5. เช็คความแม่นยำว่า AI ตัวนี้ฉลาดแค่ไหน
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"🎯 ความแม่นยำของ AI ตัวนี้อยู่ที่: {accuracy * 100:.2f}%")

# 6. เซฟสมองกลนี้เก็บไว้เป็นไฟล์ชื่อ 'asl_model.p' เพื่อเอาไปใช้แปลจริง
with open('asl_model.p', 'wb') as f:
    pickle.dump({'model': model}, f)

print(" Saved model successfully!")

 Loaded data: 381 rows
 Training AI Model...
🎯 ความแม่นยำของ AI ตัวนี้อยู่ที่: 98.70%
 Saved model successfully!


In [1]:
import cv2
import mediapipe as mp
import pickle

# 1. โหลดสมองกล AI ที่เทรนไว้ (expecting 60 features)
with open('asl_model.p', 'rb') as f:
    model_data = pickle.load(f)
    model = model_data['model']

# 2. ตั้งค่า MediaPipe
mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
hands = mp_hands.Hands(static_image_mode=False, max_num_hands=1, min_detection_confidence=0.7)

cap = cv2.VideoCapture(0)

print("🤖 ระบบแปลภาษามือเปิดใช้งานแล้ว! (กด 'q' เพื่อปิด)")

while cap.isOpened():
    success, frame = cap.read()
    if not success: break

    frame = cv2.flip(frame, 1)
    h, w, c = frame.shape
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(image_rgb)

    if results.multi_hand_landmarks:
        for hand_landmarks in results.multi_hand_landmarks:
            # วาดเส้นข้อต่อบนมือ
            mp_drawing.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
            
            # ดึงพิกัดข้อมือ (ID 0) เพื่อมาหักลบทำ Relative Coordinates เหมือนตอนเทรน
            wrist = hand_landmarks.landmark[0]
            
            current_landmarks = []
            x_pixel_list = []
            y_pixel_list = []
            
            # วนลูปเก็บพิกัดพิกเซลเอาไว้หาตำแหน่งวาดตัวหนังสือบนหน้าจอ
            for lm in hand_landmarks.landmark:
                x_pixel_list.append(int(lm.x * w))
                y_pixel_list.append(int(lm.y * h))
            
            # ดึงพิกัดจุดที่ 1 ถึง 20 มาหักลบกับข้อมือเพื่อให้ได้ 60 ค่าตามที่โมเดลต้องการ
            for i in range(1, 21):
                lm = hand_landmarks.landmark[i]
                
                rel_x = lm.x - wrist.x
                rel_y = lm.y - wrist.y
                rel_z = lm.z - wrist.z
                
                current_landmarks.extend([rel_x, rel_y, rel_z])
            
            # --- ส่งให้ AI ทำนายผล (ส่งเข้าไป 60 ค่าวัดมิติตื้นลึก) ---
            prediction = model.predict([current_landmarks])
            predicted_letter = prediction[0] 
            
            # หาพิกัดมุมบนของมือเพื่อเอาไว้แสดงตัวอักษร
            x_min, y_min = min(x_pixel_list), min(y_pixel_list)
            
            # แสดงตัวอักษรคำแปล
            cv2.putText(frame, predicted_letter, (x_min, y_min - 20), 
                        cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)

    cv2.imshow('ASL Real-time Translator', frame)
    if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()

🤖 ระบบแปลภาษามือเปิดใช้งานแล้ว! (กด 'q' เพื่อปิด)
